In [ ]:
import ijson
import json
import torch
import hdbscan
import numpy as np
import re
from transformers import AutoTokenizer, AutoModel, BigBirdModel, BigBirdTokenizer
import matplotlib.pyplot as plt

In [ ]:

with open('./Data/Bail_data.json', 'r', encoding='utf-8') as f:
    objects = ijson.items(f, 'item')
    cases = list(objects)

In [ ]:
# sample_text = '''  
# Applicant applied for Anticipatory-Bail.\nIs it a withdrawal application? No.\nAge of the accused is 40 years.\nHealth issues for the accused are None.\nThere are no past criminal records of the accused.\nStatutes mentioned in the judgement are [Section 341 IPC, Section 376 IPC, Section 506 IPC, Section 511 IPC, Section 366 r/w 34 IPC].\nPrecedents mentioned in the judgement are None.\nDetails of the incident are The accused are charged with offences related to a maid servant who was allegedly physically assaulted and raped by the accused. The maid servant was working in the house of the first accused and was allegedly tied up and assaulted by the accused while the first accused's wife was not present.\nArguments supporting the bail application are The allegations made against the first accused are baseless and false. The de facto complainant had a motive to falsely implicate the first accused in the crime. The first accused lost money from the house and had suspicions about the complainant. The complainant had raised money through immoral activities and insisted on being retained as a maid servant, failing which she would commit suicide. The first accused filed a complaint before the Circle Inspector of police, and the complainant was warned by the police not to create any problem. The complainant did not make any allegations regarding rape in the petition given to the Women's Commission and only stated that the first accused was harassing her by not paying her salary. The Women's Commission did not pass any final order in the matter. The Dy.S.P conducted a thorough investigation into the matter, and the complainant did not raise any allegation of sexual assault by the first accused during the inquiry. The private complaint was filed by the complainant after about one year and seven months, and the allegations of rape were made at a highly belated stage. The documents produced in this case will show that the complainant was deliberately making false allegations against the first accused. The first accused's wife was present in the house on the date of the alleged incident, and the first accused's daughter was also in the house. The case is falsely foisted against the first accused only to wreak vengeance since he made a complaint against the complainant to the Circle Inspector of police and she was sent out of job. The de facto complainant is an educated girl who studied up to 10th standard.\nArguments opposing the bail application are The allegations made against the petitioners are very serious in nature and the case diary reveals a deplorable state of affairs and the trauma which a helpless female member of a Scheduled Tribe has gone through at the hands of the petitioners. The de facto complainant was tied up to the cot and sexually assaulted by the accused in the most heinous manner. The first accused committed rape on her, while the others held her by force and the same act was repeated by the second accused also. Medical records reveal that the complainant was examined by the doctor on the allegations of rape and she had injuries on the private part for which medicines were applied. She also underwent psychiatric treatment since the incident had inflicted mental trauma on her. She was treated as an inpatient in the hospital for a considerable time and the incident has affected her mental balance also. The impact of the incident was severe. Medical records reveal that after the sexual assault on the complainant, she started behaving abnormally and she was treated by a psychiatrist for the mental disturbance. The statement given by the employees of the hospital including the doctor clearly reveal that the complainant was subjected to sexual assault which inflicted physical injuries on her. The independent medical evidence are more than sufficient to rule out a case of false implication. The omissions made in the earlier statements given by the victim and the failure to mention about the incident to the Women's Commission etc. are wilfully explained by the tell-tale materials in the case diary, including medical records. A reading of the private complaint itself will show that the complainant was referring to the time at which the accused came to the house. But, the incident happened much later, after she cooked the food. It cannot be said that it was 9:30 A.M that the incident happened, even on a plain reading of the FIS. It was also pointed out that in FIS, she had specifically stated that the first accused's wife was not present in the house when the incident happened. Regarding the date of occurrence, there could only be a typographical error, which will be evident from a reading of the complaint as a whole. The investigating officer had filed a report after questioning the complainant and it is stated that instead of 04/02/2008, it was recorded as 04/02/2007 in the private complaint. Regarding the omission in the complaint given to the Vanitha Commission and the Dy.S.P etc, there is sufficient explanation which is evident from the materials collected during investigation. A reading of the case diary as a whole will reveal that the allegations made against the petitioners are prima facie true and this is not a fit case to grant anticipatory bail
# '''

In [ ]:
def get_bail_type(case_text):
    pattern = r"Applicant applied for\s+([A-Za-z- ]+)\."
    match = re.search(pattern, case_text)
    if match:
        bail_type = match.group(1).strip()
        return bail_type.lower()
    return None


def get_outcome(case_text):
    pattern = r"The outcome of the case is\s+([A-Za-z- ]+)\."
    match = re.search(pattern, case_text)
    if match:
        outcome = match.group(1).strip()
        return outcome.lower()
    return None

In [ ]:
types = set()

regular_bail_cases_granted = []
anticipatory_bail_cases_granted = []
regular_bail_cases_denied = []
anticipatory_bail_cases_denied = []

mixed_case_count = 0

for info in cases:
    bail_type = get_bail_type(info['case'])
    bail_outcome = get_outcome(info['outcome'])

    if not bail_type or not bail_outcome:
        continue

    anticipatory_bail = 'anticipatory-bail' in bail_type
    regular_bail = 'regular-bail' in bail_type
    bail_granted = 'bail granted' in bail_outcome
    bail_denied = 'bail not granted' in bail_outcome

    if not (anticipatory_bail ^ regular_bail) or not (bail_granted ^ bail_denied):
        mixed_case_count += 1
        continue

    # types.add(bail_type)

    if anticipatory_bail:
        if bail_granted:
            anticipatory_bail_cases_granted.append(info)
        else:
            anticipatory_bail_cases_denied.append(info)
    elif regular_bail:
        if bail_granted:
            regular_bail_cases_granted.append(info)
        else:
            regular_bail_cases_denied.append(info)


In [ ]:
# len(regular_bail_cases_granted), len(anticipatory_bail_cases_granted), len(regular_bail_cases_denied), len(anticipatory_bail_cases_denied), mixed_case_count



In [ ]:
regular_bail_cases_granted.sort(key=lambda x: len(x['case']), reverse=True)
anticipatory_bail_cases_granted.sort(key=lambda x: len(x['case']), reverse=True)
regular_bail_cases_denied.sort(key=lambda x: len(x['case']), reverse=True)
anticipatory_bail_cases_denied.sort(key=lambda x: len(x['case']), reverse=True)

class_size = 10000
# training_data = regular_bail_cases_granted[:class_size] + anticipatory_bail_cases_granted[:class_size] + regular_bail_cases_denied[:class_size] + anticipatory_bail_cases_denied[:class_size]

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
clustering_model = 'law-ai/InLegalBERT'
c_tokenizer = AutoTokenizer.from_pretrained(clustering_model)
c_model = AutoModel.from_pretrained(clustering_model).to(device)

In [ ]:
token_counts = [len(c_tokenizer.tokenize(case['reasoning'])) for case in cases]
plt.hist(token_counts, bins=50)
plt.xlabel('Number of Tokens')
plt.ylabel('Number of Cases')
plt.title('Distribution of Token Counts in Legal Cases')
plt.show()

In [ ]:
reasoning_list = []
for case in cases:
    reasoning_list.append({
        'CNR': case['CNR'],
        'reasoning': case['reasoning']
    })

# shuffle the reasoning list to ensure randomness
import random
random.shuffle(reasoning_list)

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state  # (B, L, H)
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = (token_embeddings * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


In [ ]:
sentences = [x["reasoning"] for x in reasoning_list]
cnrs = [x["CNR"] for x in reasoning_list]

batch_size = 64 
all_embeddings = []

with torch.no_grad():
    for i in range(0, len(sentences), batch_size):
        batch_sents = sentences[i:i+batch_size]
        encoded = c_tokenizer(
            batch_sents,
            padding=True,
            truncation=True,
            max_length=400,
            return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        outputs = c_model(**encoded)
        emb = mean_pooling(outputs, encoded["attention_mask"])  # (B, H)
        all_embeddings.append(emb.cpu())

X = torch.cat(all_embeddings, dim=0).numpy()  # (N, 768)


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.9, random_state=42)
X_reduced = pca.fit_transform(X)


In [ ]:
X_reduced = X_reduced.astype(np.float32)
print(X_reduced.shape)
X_norm = X_reduced / (np.linalg.norm(X_reduced, axis=1, keepdims=True) + 1e-12)

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=30,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)

labels = clusterer.fit_predict(X_norm)

In [ ]:
clustered_output = []
for cnr, sent, lab in zip(cnrs, sentences, labels):
    clustered_output.append({
        "CNR": cnr,
        "reasoning": sent,
        "cluster": int(lab)
    })


In [ ]:
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int(np.sum(labels == -1))

print("Clusters:", n_clusters)
print("Noise points:", n_noise, f"({n_noise/len(labels)*100:.2f}%)")

In [ ]:
from collections import Counter
sizes = Counter(labels)
# Remove noise
sizes.pop(-1, None)
print("Top 10 cluster sizes:", sizes.most_common(10))


In [ ]:
from collections import Counter
import numpy as np

# sizes of clusters (exclude noise = -1)
sizes = Counter(labels)
sizes.pop(-1, None)

# top 10 biggest clusters by size
top10 = [cid for cid, _ in sizes.most_common(10)]

def show_cluster(cluster_id, k=10):
    idx = np.where(labels == cluster_id)[0][:k]
    print(f"\n=== Cluster {cluster_id} | size={sizes[cluster_id]} | showing {len(idx)} examples ===")
    for i in idx:
        snippet = sentences[i].replace("\n", " ")[:250]
        print(f"- [{cnrs[i]}] {snippet}")

for cid in top10:
    show_cluster(cid, k=10)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
from collections import defaultdict

X_pca = X_norm

cluster_stats = {}

unique_clusters = sorted(c for c in set(labels) if c != -1)

for cid in unique_clusters:
    idx = np.where(labels == cid)[0]
    Xc = X_pca[idx]

    n = len(Xc)
    if n < 2:
        continue

    # pairwise distances
    D = pairwise_distances(Xc, metric="euclidean")

    # upper triangular without diagonal
    dists = D[np.triu_indices(n, k=1)]

    cluster_stats[cid] = {
        "size": n,
        "mean_pairwise_dist": float(dists.mean()),
        "median_pairwise_dist": float(np.median(dists)),
        "p95_pairwise_dist": float(np.percentile(dists, 95))
    }

import matplotlib.pyplot as plt

df = pd.DataFrame.from_dict(cluster_stats, orient="index")

df_sorted = df.sort_values("median_pairwise_dist").reset_index(drop=True)




In [ ]:
up_cutoff = 0.8 
down_cutoff = 0.75 

vals = df_sorted["median_pairwise_dist"].values

x_down = np.searchsorted(vals, down_cutoff, side="left")
x_up   = np.searchsorted(vals, up_cutoff, side="left")

# print("x index for down_cutoff:", x_down)
# print("x index for up_cutoff:", x_up)

plt.figure(figsize=(10, 4))
plt.plot(df_sorted["median_pairwise_dist"].values, linewidth=2)
plt.axhline(up_cutoff, color="red", linestyle="--", label=f"up_cutoff = {up_cutoff}")
plt.axhline(down_cutoff, color="blue", linestyle="--", label=f"down_cutoff = {down_cutoff}")
plt.axvline(x_down, color="blue", linestyle=":", label=f"x @ down_cutoff = {x_down}")
plt.axvline(x_up, color="red", linestyle=":", label=f"x @ up_cutoff = {x_up}")
plt.legend()
plt.xlabel("Cluster rank (sorted)")
plt.ylabel("Median pairwise distance")
plt.title("Cluster dispersion — cutoff visualization")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
low_var_clusters = df[(df["median_pairwise_dist"] >= down_cutoff) & (df["median_pairwise_dist"] <= up_cutoff)].index.tolist()

print(f"Number of low-variance clusters: {len(low_var_clusters)}")
print("Some cluster IDs:", low_var_clusters[:10])

def show_cluster_sentences(cluster_id, max_examples=10):
    idx = np.where(labels == cluster_id)[0]
    print(f"\n=== Cluster {cluster_id} | size = {len(idx)} ===")
    for i in idx[:max_examples]:
        text = sentences[i].replace("\n", " ")[:300]
        print(f"- [{cnrs[i]}] {text}")


for cid in low_var_clusters[0:5]:   # inspect clusters from rank 105 to 125
    show_cluster_sentences(cid, max_examples=8)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

Xc_all = X_pca  # or X_norm, but be consistent

def show_representatives(cluster_id, k=5):
    idx = np.where(labels == cluster_id)[0]
    Xc = Xc_all[idx]

    # centroid in embedding space
    centroid = Xc.mean(axis=0, keepdims=True)

    sims = cosine_similarity(Xc, centroid).ravel()
    top = idx[np.argsort(-sims)[:k]]

    print(f"\n=== Cluster {cluster_id} | representatives ===")
    for i in top:
        print(f"- [{cnrs[i]}] {sentences[i][:300].replace('\\n',' ')}")

for cid in low_var_clusters[:5]:
    show_representatives(cid, k=5)


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["median_pairwise_dist"], bins=40)
plt.xlabel("Median pairwise distance")
plt.ylabel("Number of clusters")
plt.title("Distribution of cluster dispersion")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
outlier_idx = np.where(labels == -1)[0]
print(f"Total outliers: {len(outlier_idx)} ({len(outlier_idx)/len(labels)*100:.1f}%)")

import random
import time

def show_random_outliers(n=20, seed=None):
    if seed is None:
        seed = int(time.time())
    random.seed(seed)
    sample_idx = random.sample(list(outlier_idx), min(n, len(outlier_idx)))

    print(f"\n=== Showing {len(sample_idx)} random HDBSCAN outliers ===\n")
    for i in sample_idx:
        text = sentences[i].replace("\n", " ")[:400]
        print(f"- [{cnrs[i]}] {text}\n")

show_random_outliers(n=20)



In [ ]:
import matplotlib.pyplot as plt

clustered_idx = np.where(labels != -1)[0]
lens_out = [len(sentences[i].split()) for i in outlier_idx]
lens_in  = [len(sentences[i].split()) for i in clustered_idx]

plt.figure(figsize=(8,4))
plt.hist(lens_out, bins=50, alpha=0.6, label="Outliers")
plt.hist(lens_in, bins=50, alpha=0.6, label="Clustered")
plt.xlabel("Sentence length (tokens)")
plt.ylabel("Count")
plt.legend()
plt.title("Sentence length: outliers vs clustered")
plt.tight_layout()
plt.show()


In [ ]:
bad_clusters = df[df["median_pairwise_dist"] < 0.8].index.tolist()
print("Low-variance clusters:", bad_clusters)
print("Count:", len(bad_clusters))

bad_mask = np.isin(labels, bad_clusters)   # True for points in those clusters
bad_idx = np.where(bad_mask)[0]
print("Num datapoints to remove:", len(bad_idx))

bad_cnrs = [cnrs[i] for i in bad_idx]
# or as a set if you want unique IDs
bad_cnrs_set = set(bad_cnrs)

print("Unique CNRs to remove:", len(bad_cnrs_set))

In [ ]:
withdrawal_cases = []
for d in cases:
    case_text = d['case']
    withdrawal_pattern = r"Is it a withdrawal application\?\s*([^\.]+)\."

    match = re.search(withdrawal_pattern, case_text)
    if match:
        answer = match.group(1).strip()
        if answer.lower() == "yes":
            withdrawal_cases.append(d['CNR'])

In [ ]:
bad_cnrs_set.update(withdrawal_cases)

In [ ]:
good_cases = []
for case in cases:
    if case['CNR'] not in bad_cnrs_set:
        good_cases.append(case)



In [ ]:
len(good_cases)

In [ ]:
with open('Data/good_reasoning_cases.json', 'w', encoding='utf-8') as f:
    json.dump(good_cases, f, indent=4, ensure_ascii=False)

In [ ]:
types = set()

regular_bail_cases_granted = []
anticipatory_bail_cases_granted = []
regular_bail_cases_denied = []
anticipatory_bail_cases_denied = []

mixed_case_count = 0

for info in good_cases:
    bail_type = get_bail_type(info['case'])
    bail_outcome = get_outcome(info['outcome'])

    if not bail_type or not bail_outcome:
        continue

    anticipatory_bail = 'anticipatory-bail' in bail_type
    regular_bail = 'regular-bail' in bail_type
    bail_granted = 'bail granted' in bail_outcome
    bail_denied = 'bail not granted' in bail_outcome

    if not (anticipatory_bail ^ regular_bail) or not (bail_granted ^ bail_denied):
        mixed_case_count += 1
        continue

    # types.add(bail_type)

    if anticipatory_bail:
        if bail_granted:
            anticipatory_bail_cases_granted.append(info)
        else:
            anticipatory_bail_cases_denied.append(info)
    elif regular_bail:
        if bail_granted:
            regular_bail_cases_granted.append(info)
        else:
            regular_bail_cases_denied.append(info)

In [ ]:
# len(regular_bail_cases_granted), len(anticipatory_bail_cases_granted), len(regular_bail_cases_denied), len(anticipatory_bail_cases_denied), mixed_case_count
# Save the categorized cases to JSON files
with open('Data/regular_accepted.json', 'w', encoding='utf-8') as f:
    json.dump(regular_bail_cases_granted, f, indent=4, ensure_ascii=False)

with open('Data/anticipatory_accepted.json', 'w', encoding='utf-8') as f:
    json.dump(anticipatory_bail_cases_granted, f, indent=4, ensure_ascii=False)

with open('Data/regular_denied.json', 'w', encoding='utf-8') as f:
    json.dump(regular_bail_cases_denied, f, indent=4, ensure_ascii=False)

with open('Data/anticipatory_denied.json', 'w', encoding='utf-8') as f:
    json.dump(anticipatory_bail_cases_denied, f, indent=4, ensure_ascii=False)

print(f"Saved {len(regular_bail_cases_granted)} regular accepted cases")
print(f"Saved {len(anticipatory_bail_cases_granted)} anticipatory accepted cases")
print(f"Saved {len(regular_bail_cases_denied)} regular denied cases")
print(f"Saved {len(anticipatory_bail_cases_denied)} anticipatory denied cases")


In [ ]:
# # dump training data to balanced_training_data.json

# with open('balanced_training_data.json', 'w', encoding='utf-8') as f:
#     json.dump(training_data, f, indent=4, ensure_ascii=False)


In [ ]:
with open('./Data/balanced_training_data.json', 'r', encoding='utf-8') as f:
    objects = ijson.items(f, 'item')
    cases = list(objects)

In [ ]:
tokenizer = BigBirdTokenizer.from_pretrained("google/bigbird-roberta-large")

In [ ]:
# create a bar chart of no. of tokens in each case

token_counts = [len(tokenizer.tokenize(case['case'])) for case in good_cases]
plt.hist(token_counts, bins=50)
plt.xlabel('Number of Tokens')
plt.ylabel('Number of Cases')
plt.title('Distribution of Token Counts in Legal Cases')
plt.show()